# 🦊 Fox-Mecha Super Brain LoRA Trainer (Universal CPU/GPU)
Train Fox-Mecha on Qwen2.5-0.5B.
Works on **Free Colab CPU** (no GPU quota needed!) or GPU.
Time: ~5-7 minutes on Colab CPU / ~2 minutes on GPU.
Output: ~10MB `fox_lora_10mb.zip`

In [ ]:
# 1. Install Libraries
!pip uninstall -y torchao
!pip install -q transformers peft datasets accelerate
print("✅ Libraries installed successfully!")

In [ ]:
# 2. Load Model, Dataset and Train
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Running training on: {device.upper()}")

# Check dataset file
if not os.path.exists("fox_bengali_dataset.jsonl"):
    raise FileNotFoundError("❌ fox_bengali_dataset.jsonl file not found! Please upload it via the left Files panel.")

print("📥 Loading Qwen2.5-0.5B tokenizer & model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
)

# Apply LoRA (Rank 16 produces ~10MB adapter)
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Load and tokenize dataset
print("📖 Processing dataset...")
dataset = load_dataset("json", data_files="fox_bengali_dataset.jsonl", split="train")

def tokenize_func(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, truncation=True, max_length=256)
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_dataset = dataset.map(tokenize_func, remove_columns=dataset.column_names)

# Training Configuration (Fast dynamic batching for CPU & GPU)
training_args = TrainingArguments(
    output_dir="./fox_lora_checkpoints",
    per_device_train_batch_size=4 if device == "cuda" else 2,
    gradient_accumulation_steps=2 if device == "cuda" else 4,
    learning_rate=3e-4,
    max_steps=100,
    logging_steps=10,
    save_strategy="no",
    fp16=(device == "cuda"),
    bf16=False,
    optim="adamw_torch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8),
    args=training_args,
)

print("🚀 Starting Training (100 steps)... Please wait a few minutes!")
trainer.train()
print("🎉 Training completed successfully!")


In [ ]:
# 3. Save LoRA adapter and download (~10MB)
import shutil
import os

save_dir = "./fox_lora_final"
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

zip_filename = "fox_lora_10mb"
shutil.make_archive(zip_filename, "zip", save_dir)
zip_file = f"{zip_filename}.zip"

file_size_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f"✅ Successfully generated {zip_file} ({file_size_mb:.2f} MB)!")

try:
    from google.colab import files
    files.download(zip_file)
    print("⬇️ Download popup initiated!")
except Exception as e:
    print(f"Files ready. You can manually download {zip_file} from the left files tab.")